# CML 2 - Data preparation for ML

**Scaler CML | Applied ML (Intro)**  
**Dataset:** California housing - predict `median_house_value`.

Companion notes: `Student_Notes_CML2_Data_Prep.md`

**Today's spine:** hook fake-good scores → split first → impute / encode / scale without leakage → `ColumnTransformer` + `Pipeline` → evaluate.

File: `data/california_housing.csv`


## 0. Imports and load

Import `pandas`, `numpy`, and the sklearn tools you expect to need.

Load `data/california_housing.csv` into a DataFrame `df`.

Print the shape and the first five rows.

**Discuss:** Before looking at metrics, what would make you distrust a "perfect" score on this table?


In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler

df = pd.read_csv("data/california_housing.csv")
print(df.shape)
df.head()


## 1. First look

Get oriented before transforming anything.

1. What is the target column?
2. Which columns are numeric? Which are not?
3. Summarize `median_house_value` (mean, median, min, max).
4. How many unique values does `ocean_proximity` have? List them.

**Discuss:** Which column looks like a good *nominal* categorical for one-hot encoding?


In [ ]:
target = "median_house_value"
print("Target:", target)

num_like = df.select_dtypes(include=[np.number]).columns.tolist()
cat_like = df.select_dtypes(exclude=[np.number]).columns.tolist()
print("Numeric:", num_like)
print("Non-numeric:", cat_like)

print(df[target].agg(["mean", "median", "min", "max"]))

print("ocean_proximity nunique:", df["ocean_proximity"].nunique())
print(df["ocean_proximity"].unique())


## 2. Missing values (inventory)

1. Count missing values per column. Which column(s) have NaNs?
2. What % of rows are missing that field?
3. Would you drop those rows, drop the column, or impute? Why?

**Discuss (60s pair):** For ~1% missing vs ~40% missing, when does "drop rows" stop being free?


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (100 * missing / len(df)).round(2)
print(pd.DataFrame({"n_missing": missing, "pct": missing_pct}).query("n_missing > 0"))

# Rough guide for this dataset: ~1% missing in total_bedrooms.
# Prefer median impute (after the split) over dropping the whole column.


## 3. Feature set for today

Keep a readable subset. Build `data` with:

| Column | Role |
|---------|-------|
| `median_house_value` | target |
| `median_income` | numeric |
| `housing_median_age` | numeric |
| `total_rooms` | numeric |
| `total_bedrooms` | numeric (has missing) |
| `population` | numeric |
| `households` | numeric |
| `ocean_proximity` | categorical (nominal) |

Also create an **ordinal** column `income_band` from `median_income` using three ordered labels (`low` < `mid` < `high`).

Use **fixed cut points** (not full-frame `qcut`) so band edges do not peek at held-out districts.

Check `data.shape` and missing counts on this subset.

**Discuss:** Why is inventing `income_band` useful for class even if `median_income` stays as a numeric feature?


In [ ]:
cols = [
    "median_house_value",
    "median_income",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "ocean_proximity",
]
data = df[cols].copy()

# Fixed cuts (approx tertile-ish on CA housing income) - no full-frame quantile fit.
data["income_band"] = pd.cut(
    data["median_income"],
    bins=[-np.inf, 2.5, 4.5, np.inf],
    labels=["low", "mid", "high"],
)

print(data.shape)
print(data.isna().sum())
print(data["income_band"].value_counts().sort_index())


## 4. Features and target

Create `X` and `y`.

Define three column lists:
- `num_cols`
- `nom_cols` (nominal)
- `ord_cols` (ordinal - at least `income_band`)

Should `median_income` stay in `num_cols` if you already built `income_band` from it? Decide and note why.

**Discuss:** If you kept both, what kind of redundancy are you accepting for teaching ordinal encoding?


In [ ]:
y = data["median_house_value"]
X = data.drop(columns=["median_house_value"])

# Keep median_income: continuous signal is stronger than 3 bands alone.
# income_band is here mainly to practice ordinal encoding (some redundancy is OK for class).
num_cols = [
    "median_income",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
]
nom_cols = ["ocean_proximity"]
ord_cols = ["income_band"]

print(X.shape, y.shape)
print("num:", num_cols)
print("nom:", nom_cols)
print("ord:", ord_cols)


## 5. Split first

Create `X_train`, `X_test`, `y_train`, `y_test` (80/20, `random_state=42`).

Which of these is safe **before** the split?

- A) Fit a scaler on all of `X`
- B) Drop rows where the target is missing
- C) Fill `total_bedrooms` NaNs with the median of the full frame

Write your choice as a short comment, then run the split.

**Class rule:** any statistic learned from data (mean, std, category list, median fill) must be learned from train only, then applied to test.

**Discuss:** Why does "I scaled the whole CSV then split" feel efficient but still leak?


In [ ]:
# Your answer (A / B / C): write a one-line comment here before running.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)


## 6. Leakage demo - wrong order

Using the **full** numeric part of `X` (before relying on your train/test split):

1. Fit `StandardScaler` on all rows.
2. Transform all rows.
3. Then `train_test_split` the scaled matrix.

Compare the scaler's stored mean for `total_rooms` with the mean of `total_rooms` on the held-out houses from section 5.

**Discuss:** Did the scaler see the test districts? What goes wrong when you trust the test score after this?

Hook line: prep mistakes look like fake good scores, not stack traces.


In [ ]:
X_num_all = X[num_cols].copy()

leaky_scaler = StandardScaler()
X_num_scaled_all = leaky_scaler.fit_transform(X_num_all)

X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(
    X_num_scaled_all, y, test_size=0.2, random_state=42
)

rooms_idx = num_cols.index("total_rooms")
print("Scaler mean_ (total_rooms) from ALL rows:", leaky_scaler.mean_[rooms_idx])
print("True mean total_rooms on section-5 X_test:", X_test["total_rooms"].mean())
print("True mean total_rooms on ALL X:", X["total_rooms"].mean())
print("True mean total_rooms on section-5 X_train:", X_train["total_rooms"].mean())

# Compare the three means. If scaler mean matches ALL (and is pulled toward test),
# the transform already peeked at held-out districts.


## 7. Impute - fit on train only

1. How many `total_bedrooms` NaNs are in train vs test?
2. Fit `SimpleImputer(strategy="median")` on the train numeric columns.
3. Transform train and test with that same imputer.

**Discuss:** Why must you not call `.fit` again on the test set?

Remember Quiz 3 energy: for skewed numerics with outliers, median fill is often stabler than mean fill.


In [ ]:
print("NaNs train:", X_train["total_bedrooms"].isna().sum())
print("NaNs test:", X_test["total_bedrooms"].isna().sum())

num_imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[num_cols]),
    columns=num_cols,
    index=X_train.index,
)
X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[num_cols]),
    columns=num_cols,
    index=X_test.index,
)

print("median used for total_bedrooms:", num_imputer.statistics_[num_cols.index("total_bedrooms")])
print("NaNs after impute - train:", X_train_num.isna().sum().sum(), "test:", X_test_num.isna().sum().sum())


## 8. Encoding (nominal vs ordinal)

1. Fit a one-hot encoder on train `ocean_proximity` with `handle_unknown="ignore"`. Transform train and test.
2. Fit an ordinal encoder on train `income_band` with explicit order `low` < `mid` < `high`. Transform train and test.

Questions to answer in comments:

- What breaks if you label-encode `ocean_proximity` as 0,1,2,… for a distance-based model?
- A new ocean category appears only in test - what does `handle_unknown="ignore"` do?

**Discuss:** Card vote - Red/Blue/Green vs Low/Med/High. Which gets one-hot? Which gets ordered codes?


In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_nom = ohe.fit_transform(X_train[nom_cols])
X_test_nom = ohe.transform(X_test[nom_cols])
print("OHE feature names:", ohe.get_feature_names_out(nom_cols))
print("train OHE shape:", X_train_nom.shape, "test OHE shape:", X_test_nom.shape)

ord_enc = OrdinalEncoder(categories=[["low", "mid", "high"]])
X_train_ord = ord_enc.fit_transform(X_train[ord_cols])
X_test_ord = ord_enc.transform(X_test[ord_cols])
print("ordinal categories_", ord_enc.categories_)
print("sample train ordinal:", X_train_ord[:5].ravel())

# Comment your answers to the two questions above.


## 9. Scaling

On imputed numeric train features:

1. Print mean and std of `median_income` and `total_rooms` before scaling.
2. Fit `StandardScaler` on train; transform train and test.
3. Print mean/std of the scaled train columns - what should they be near?

**Discuss:** Without scaling, which feature would dominate Euclidean distance? Why?

Quick map:

| Scaler | Idea | When |
|---|---|---|
| Standard | mean 0, std 1 | Default for many linear / distance methods |
| MinMax | [0, 1] | Bounded ranges |
| Robust | median / IQR | Heavy outliers |


In [ ]:
print("Before scale:")
print(X_train_num[["median_income", "total_rooms"]].agg(["mean", "std"]))

scaler = StandardScaler()
X_train_num_s = pd.DataFrame(
    scaler.fit_transform(X_train_num),
    columns=num_cols,
    index=X_train_num.index,
)
X_test_num_s = pd.DataFrame(
    scaler.transform(X_test_num),
    columns=num_cols,
    index=X_test_num.index,
)

print("\nAfter scale (train) - expect mean~0, std~1:")
print(X_train_num_s[["median_income", "total_rooms"]].agg(["mean", "std"]).round(6))


## 10. Optional - RobustScaler

`total_rooms` and `population` can have heavy tails.

Fit a `RobustScaler` on train numerics and compare a few rows to `StandardScaler`.

**Discuss:** When would you prefer robust scaling over z-score?


In [ ]:
robust = RobustScaler()
X_train_num_r = pd.DataFrame(
    robust.fit_transform(X_train_num),
    columns=num_cols,
    index=X_train_num.index,
)

compare = pd.DataFrame({
    "raw_total_rooms": X_train_num["total_rooms"].head(5).values,
    "standard": X_train_num_s["total_rooms"].head(5).values,
    "robust": X_train_num_r["total_rooms"].head(5).values,
})
print(compare)


## 11. Pipeline - glue without forgetting fit-on-train

Build one `ColumnTransformer` + `Pipeline`:

- numeric: median impute → `StandardScaler`
- nominal (`ocean_proximity`): most-frequent impute → one-hot (`handle_unknown="ignore"`)
- ordinal (`income_band`): most-frequent impute → `OrdinalEncoder` with your order

Attach `Ridge`. Fit once on `X_train`, `y_train`. Predict on `X_test`.

**Line to land:** Pipeline is the practical way to not forget fit-on-train.

**Discuss:** What does a single `.fit()` on the full pipeline lock inside the TRAIN STATS box?


In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[["low", "mid", "high"]])),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("nom", nominal_pipe, nom_cols),
    ("ord", ordinal_pipe, ord_cols),
])

model = Pipeline([
    ("prep", preprocess),
    ("ridge", Ridge(alpha=1.0)),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Predictions ready:", y_pred[:5])


## 12. Evaluate

On the test set, report RMSE, MAE, and \(R^2\).

**Discuss:** If test \(R^2\) looks unrealistically high after you fitted prep on all rows, what should you suspect?

Preview: Session 3 revisits scaling when gradient descent stability shows up.


In [ ]:
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"R^2:  {r2:.4f}")


## 13. Leakage detective

For each case, write `Leak` or `No leak` in a comment below. Fix the leaky ones in code if you have time.

1. Fill `total_bedrooms` with the median of the full `df`, then split and train.
2. Split first. Fit one-hot on train `ocean_proximity`; transform test (`handle_unknown="ignore"`).
3. Add `value_bin = (median_house_value > median)` into `X`, then predict `median_house_value`.
4. Put the highest-income 20% of districts into the test set on purpose, then claim a general test score.

**Discuss:** For case 3, use the prediction-time test: would you already have that number when scoring a new district?


In [ ]:
# 1) ...
# 2) ...
# 3) ...
# 4) ...

# Optional fixes (attempt after you label each case):
# - Impute AFTER split (fit on train only)
# - Never put label-derived columns into X
# - Prefer random (or time-based) holdout, not cherry-picked rich districts

fix_imputer = SimpleImputer(strategy="median")
X_train_fix = X_train.copy()
X_test_fix = X_test.copy()
X_train_fix[num_cols] = fix_imputer.fit_transform(X_train[num_cols])
X_test_fix[num_cols] = fix_imputer.transform(X_test[num_cols])
print("Train-only imputer ready; label cases 1-4 in the comments above.")


## 14. Exit check (unmarked)

Answer in comments or `print` strings (no looking up solutions):

1. When are you allowed to compute mean / median / category lists from the data?
2. Name one nominal and one ordinal column from today's work.
3. Why put impute → encode/scale → model inside a `Pipeline`?

Also from the paper exit ticket:
- One sentence: what is preprocessing leakage?
- Name two transforms that must be fit on train only.


In [ ]:
# Write your exit answers here (student materials stay unmarked).
print("Exit check: fill in answers as comments or edit these strings.")
print("1) ...")
print("2) ...")
print("3) ...")
